# 02 — Run the pipeline on the local sample reports

Ingests whatever report files currently sit in `data/raw`, parses them, runs the classifier, and prints probabilities alongside the evidence sentence that drove each label — for manually eyeballing predictions on the small local sample set.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
import pandas as pd

from report2label.ingestion import read_report_file
from report2label.parsing.report_parser import parse_report
from report2label.extraction.label_mapper import LabelVocabulary
from report2label.extraction.predictor import LabelPredictor
from report2label.extraction.thresholding import resolve_thresholds
from report2label.models.model_loader import ModelConfig
from report2label.utils.io import list_report_files, read_yaml

pipeline_config = read_yaml(PROJECT_ROOT / "configs" / "pipeline.yaml")
model_config = ModelConfig.from_yaml(PROJECT_ROOT / "configs" / "model.yaml")
label_vocab = LabelVocabulary.from_yaml(PROJECT_ROOT / "configs" / "labels.yaml")
thresholds = resolve_thresholds(
    label_vocab.names, model_config.default_threshold, pipeline_config.get("label_thresholds")
)
predictor = LabelPredictor(model_config, label_vocab, thresholds, pipeline_config["evidence"])

c:\Users\pramu\Desktop\FYP\Project-Codebase\ChestCT-Report2Label\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-18 14:59:50 [INFO] report2label.models.model_loader: Loading tokenizer/encoder 'zzxslp/RadBERT-RoBERTa-4m' on cpu
2026-09-18 14:59:51 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/zzxslp/RadBERT-RoBERTa-4m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-18 14:59:51 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/zzxslp/RadBERT-RoBERTa-4m/b8b7433023c433567a1b3e6fc3fede831b806e75/config.json?%2Fzzxslp%2FRadBERT-RoBERTa-4m%2Fresolve%2Fmain%2Fconfig.json=&etag=%223e7845be874d0101ce4bbb1f92981d9a9a785e2a%22 "HTTP/1.1 200 OK"
2026-09-18 14:59:52 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/zzxslp/RadBERT-RoBERTa-4m/resolve/

In [3]:
raw_dir = PROJECT_ROOT / pipeline_config["paths"]["raw_reports_dir"]
files = list_report_files(raw_dir, pipeline_config["ingestion"]["supported_extensions"])
print(f"Found {len(files)} report(s) under {raw_dir}")

parsed_reports = []
for file_path in files:
    raw_text = read_report_file(file_path)
    parsed = parse_report(
        raw_text,
        section_headers=pipeline_config["sections"]["headers"],
        classifier_input_sections=pipeline_config["sections"]["classifier_input_sections"],
        phi_removal_enabled=pipeline_config["phi_removal"]["enabled"],
        source_path=str(file_path),
    )
    parsed_reports.append(parsed)
    print(f"{file_path.name}: ct_id={parsed.ct_id}")

Found 7 report(s) under c:\Users\pramu\Desktop\FYP\Project-Codebase\ChestCT-Report2Label\data\raw
4203.pdf: ct_id=4203/26


2026-09-18 14:59:56 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/zzxslp/RadBERT-RoBERTa-4m/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"


4213.pdf: ct_id=4213/26
4214.pdf: ct_id=4214/26
4215.pdf: ct_id=4215/26
4216.pdf: ct_id=4216/26
4217.pdf: ct_id=4217
4222.pdf: ct_id=4222


2026-09-18 14:59:56 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/zzxslp/RadBERT-RoBERTa-4m/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"


In [4]:
predictions = [
    predictor.predict(p.classifier_text, ct_id=p.ct_id, source_path=p.source_path)
    for p in parsed_reports
]

summary = pd.DataFrame(
    [{"report": Path(p.source_path).name, **pred.probabilities} for p, pred in zip(parsed_reports, predictions)]
)
summary.set_index("report")

,Medical material,Arterial wall calcification,Cardiomegaly,Pericardial effusion,Coronary artery wall calcification,Hiatal hernia,Lymphadenopathy,Emphysema,Atelectasis,Lung nodule,Lung opacity,Pulmonary fibrotic sequela,Pleural effusion,Mosaic attenuation pattern,Peribronchial thickening,Consolidation,Bronchiectasis,Interlobular septal thickening
report,,,,,,,,,,,,,,,,,,
4203.pdf,0.003639,0.000901,0.015312,0.006781,0.002981,0.002378,0.022330,0.004630,0.998024,0.010970,0.002438,0.001003,0.004914,0.847413,0.002730,0.005088,0.002164,0.003257
4213.pdf,0.030587,0.001819,0.034943,0.010195,0.010495,0.009346,0.899826,0.007667,0.004652,0.043743,0.773741,0.071219,0.001350,0.524583,0.987023,0.006383,0.026912,0.974498
4214.pdf,0.058964,0.001436,0.023823,0.096683,0.002324,0.011240,0.876028,0.211190,0.961136,0.976721,0.013009,0.991522,0.999143,0.006327,0.017519,0.032260,0.009925,0.588729
4215.pdf,0.001365,0.000651,0.000524,0.000846,0.000873,0.001177,0.972192,0.000877,0.000918,0.999051,0.001023,0.001623,0.000871,0.001552,0.001134,0.001084,0.000710,0.000725
4216.pdf,0.013822,0.004269,0.003927,0.007921,0.004141,0.003682,0.058769,0.007113,0.012630,0.010355,0.764049,0.008659,0.003770,0.039227,0.652673,0.996576,0.999314,0.015930
4217.pdf,0.008895,0.005800,0.002673,0.005379,0.002268,0.005582,0.049644,0.015514,0.024681,0.019385,0.236302,0.341435,0.003624,0.005655,0.030956,0.998089,0.999370,0.015377
4222.pdf,0.002246,0.002440,0.003217,0.005293,0.001642,0.004734,0.041246,0.004907,0.002586,0.005888,0.504208,0.026438,0.001259,0.006815,0.354078,0.009135,0.999592,0.153762


In [5]:
# Evidence for every predicted-positive label, across all reports.
for prediction in predictions:
    positive_labels = [name for name, value in prediction.labels.items() if value == 1]
    print(f"=== {Path(prediction.source_path).name} (ct_id={prediction.ct_id}) ===")
    print(f"positive labels: {positive_labels}")

    for label in positive_labels:
        print(f"\n[{label}] p={prediction.probabilities[label]:.3f}")
        for item in (prediction.evidence or {}).get(label, []):
            print(f"    ({item['probability']:.3f}) {item['sentence']}")
    print()


=== 4203.pdf (ct_id=4203/26) ===
positive labels: ['Atelectasis', 'Mosaic attenuation pattern']

[Atelectasis] p=0.998
    (1.000) Segmental atelectasis of right middle lobe and left lingular lobe
    (0.086) Mosaic attenuation in bilateral lungs involving all zones.
    (0.003) The right ventricle is mildly dilated.

[Mosaic attenuation pattern] p=0.847
    (1.000) Mosaic attenuation in bilateral lungs involving all zones.
    (0.020) Middle lobar syndrome.
    (0.008) Early pulmonary hypertension.

=== 4213.pdf (ct_id=4213/26) ===
positive labels: ['Lymphadenopathy', 'Lung opacity', 'Mosaic attenuation pattern', 'Peribronchial thickening', 'Interlobular septal thickening']

[Lymphadenopathy] p=0.900
    (1.000) Multiple enlarged mediastinal lymphnodes,largest 14x19 mm in prevascular region.
    (0.049) Impression: Mild interval progression of known fibrotic NSIP with coexisting OP.
    (0.041) Findings: Bilateral lower lobe predominant peribronchovascular intra and interlobular septa